In [1]:
# --- IMPORTACIONES ---
import yaml
from langchain_core.prompts import ChatPromptTemplate

# Importaciones propias de tu proyecto
from esquemas import LLMScaffoldingOutput
from configuracion_llm import get_llm_estructurado

# Importación de OCR (Exactamente igual que en motor_evaluacion.py)
try:
    from utils_easy import extract_text_from_image
except ImportError:
    def extract_text_from_image(path): 
        return "[Texto simulado del OCR porque utils_easy no está disponible]"

/home/ahervas2/practicas_uv/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# --- CARGA DE CONFIGURACIÓN Y MODELO ---
# 1. Cargar el prompt desde el archivo YAML
try:
    with open("prompts.yaml", "r", encoding="utf-8-sig") as f:
        prompts = yaml.safe_load(f)
    prompt_texto = prompts["generador_pistas_prompt"]
    print("✅ Prompt cargado correctamente desde prompts.yaml")
except Exception as e:
    print(f"❌ Error crítico al cargar prompts.yaml: {e}")

# 2. Instanciar el LLM con salida estructurada (Pydantic)
try:
    llm_generador = get_llm_estructurado("generador")
    
    # 3. Crear la cadena (Chain)
    prompt_tmpl = ChatPromptTemplate.from_template(prompt_texto)
    cadena = prompt_tmpl | llm_generador
    print("✅ Cadena LLM configurada y lista para generar.")
except Exception as e:
    print(f"❌ Error al configurar el LLM: {e}")

✅ Prompt cargado correctamente desde prompts.yaml
✅ Cadena LLM configurada y lista para generar.


In [ ]:
# --- DATOS DE PRUEBA ---
# Prueba cambiando la estrategia a "Navegar" o "Integrar" para ver cómo cambia el enfoque de las pistas.

datos_prueba = {
        "contexto_tarea": "El alumno debe comparar un comentario de instagram con varias respuestas para decidir qué postura es más acertada sobre el impacto ambiental.",
        "enunciado_base": "He buscado en internet “Sierra Calderona” y esto es lo que he encontrado. Docente: ¿Qué página es mejor para informarnos sobre el tema? Justifica tu respuesta de manera breve y con tus propias palabras",
        "estrategia": "evaluar",
        "subestrategia": "Analizar expertise del autor"
    }

In [9]:
tarea_ejemplo = {
    "task_id": "cien_s1_t09",
    "task_prompt": "He buscado en internet “Sierra Calderona” y esto es lo que he encontrado. Docente: ¿Qué página es mejor para informarnos sobre el tema? Justifica tu respuesta de manera breve y con tus propias palabras",
    "task_text": None,
    "task_image": ["./data/img/sierra_calderona.png"],
    "strategy": "evaluar",
    "sub_strategy": "Analizar expertise del autor"
}

context_parts = [tarea_ejemplo.get("task_text", "")]
task_images = tarea_ejemplo.get("task_image", [])

if not isinstance(task_images, list):
    task_images = [task_images]
    
for img in task_images:
    if img: 
        context_parts.append(extract_text_from_image(img))

contexto_extraido = "\n\n".join(filter(None, context_parts))

print("📄 Contexto extraído de la tarea:")
print(contexto_extraido[:200] + "...\n" if len(contexto_extraido) > 200 else contexto_extraido)

# --- 3. PREPARAMOS LOS DATOS PARA EL LLM ---
datos_prueba = {
    "contexto_extraido": contexto_extraido,
    "enunciado_base": tarea_ejemplo.get("task_prompt"),
    "estrategia": tarea_ejemplo.get("strategy"),
    "subestrategia": tarea_ejemplo.get("sub_strategy")
}

📄 Contexto extraído de la tarea:
Gs-gle
Sierra calderona
X !
2
Q
Wikipedia
htos Iles wikipedia org
wiki , Sierra
Calderona
Sierra Calderona
Wikipedia; la enciclopedia libre
Forma una alineación montañosa de orientación noroeste
sudes...



In [ ]:
# --- EJECUCIÓN Y ENSAMBLAJE ---
print("🤖 Consultando al LLM para generar las pistas...\n")

try:
    # Invocamos la cadena con los datos de prueba
    resultado = cadena.invoke(datos_prueba)
    
    # Ensamblamos y mostramos los 3 niveles finales
    print("="*50)
    print("🎯 RESULTADOS DEL ENSAMBLAJE")
    print("="*50)
    
    print("\n🟢 NIVEL ALTO (Sin pistas):")
    print(f"> {datos_prueba['enunciado_base']}")
    
    print("\n🟠 NIVEL MEDIO (Enunciado + Pista 1):")
    print(f"> {datos_prueba['enunciado_base']}\n> Pista 1:{resultado.pista_1}")
    
    print("\n🔴 NIVEL BAJO (Enunciado + Pista 1 + Pista 2):")
    print(f"> {datos_prueba['enunciado_base']}\n> Pista 1: {resultado.pista_1} Pista 2: {resultado.pista_2}")
    
except Exception as e:
    print(f"❌ Error durante la generación: {e}")

🤖 Consultando al LLM para generar las pistas...

🎯 RESULTADOS DEL ENSAMBLAJE

🟢 NIVEL ALTO (Sin pistas):
> He buscado en internet “Sierra Calderona” y esto es lo que he encontrado. Docente: ¿Qué página es mejor para informarnos sobre el tema? Justifica tu respuesta de manera breve y con tus propias palabras

🟠 NIVEL MEDIO (Enunciado + Pista 1):
> He buscado en internet “Sierra Calderona” y esto es lo que he encontrado. Docente: ¿Qué página es mejor para informarnos sobre el tema? Justifica tu respuesta de manera breve y con tus propias palabras
> 💡 Ayuda: Recuerda que en la red cualquiera puede crear una página; por eso es importante reconocer que no todos los sitios provienen de fuentes con autoridad o experiencia en el tema.

🔴 NIVEL BAJO (Enunciado + Pista 1 + Pista 2):
> He buscado en internet “Sierra Calderona” y esto es lo que he encontrado. Docente: ¿Qué página es mejor para informarnos sobre el tema? Justifica tu respuesta de manera breve y con tus propias palabras
> 💡 Ayuda: R